# CP2 — Monitor de Atenção/Fadiga (OpenCV + MediaPipe)

**Objetivo**: montar um pipeline em tempo real (webcam) para estimar **atenção / fadiga**, usando **Face Landmarker (MediaPipe Tasks)**.




## 1) Importando bibliotecas

Vamos começar do básico: importar OpenCV, NumPy e MediaPipe e checar versões.


In [1]:
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path

print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)
print('MediaPipe:', mp.__version__)


OpenCV: 4.13.0
NumPy: 2.4.4
MediaPipe: 0.10.33


## 2) Baixando o modelo do MediaPipe (Face Landmarker)

Aqui, o nosso script já tem a função `garantir_modelo()` que baixa o arquivo `face_landmarker.task` na primeira execução (se ainda não existir).


In [2]:
import attention_monitor as am

model_path = Path(am.MODEL_FILENAME)
am.garantir_modelo(model_path)
print('Modelo OK em:', model_path.resolve())


Modelo OK em: C:\ProjetoCognitive\cp2-4sir-tb\face_landmarker.task


## 3) Lendo um frame da webcam

Aqui é um padrão bem comum: **captura → checa se veio → mostra propriedades**.


In [3]:
cap = cv2.VideoCapture(0)
ok, frame = cap.read()
cap.release()

print('Capturou frame?', ok)
if ok:
    print('Shape (H,W,C):', frame.shape)
    print('dtype:', frame.dtype)


Capturou frame? True
Shape (H,W,C): (480, 640, 3)
dtype: uint8


## 4) Rodando o Face Landmarker em um frame (1 rosto)

A lógica é:
- BGR (OpenCV) → RGB
- criar `mp.Image`
- `detect_for_video(...)`
- se detectou, pegar landmarks e calcular métricas (**EAR**, **MAR**, **pose**)


In [4]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(model_path)),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    output_face_blendshapes=True,
)

if not ok:
    raise RuntimeError('Não consegui capturar frame da webcam (índice 0).')

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

with FaceLandmarker.create_from_options(options) as landmarker:
    # timestamp em ms (no modo VIDEO precisa ser crescente)
    result = landmarker.detect_for_video(mp_image, 0)

print('Tem face_landmarks?', bool(result.face_landmarks))
if result.face_landmarks:
    lm = result.face_landmarks[0]
    h, w = frame.shape[:2]
    pts = am.landmarks_para_pixels(lm, w, h)
    ear_l = am.razao_aspecto_olho(pts, am.LEFT_EYE_IDX)
    ear_r = am.razao_aspecto_olho(pts, am.RIGHT_EYE_IDX)
    ear_avg = (ear_l + ear_r) / 2.0
    mar = am.razao_aspecto_boca(pts)
    pitch, yaw, roll = am.pose_cabeca_graus(pts, w, h)
    print('EAR L/med/R:', ear_l, ear_avg, ear_r)
    print('MAR:', mar)
    print('Pose (deg) pitch/yaw/roll:', pitch, yaw, roll)


Tem face_landmarks? False


## 5) Desenhando overlay (malha) e exibindo o frame

Igual nos labs: **desenha e vê**.

> Se `cv2.imshow` não funcionar no notebook, pule esta célula e vá para a seção final (rodar o `.py`).


In [5]:
if result.face_landmarks:
    display = frame.copy()
    am.desenhar_malha_rosto(display, result.face_landmarks[0], mirror=False)
    cv2.imshow('Frame + overlay', display)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


## 6) Tempo real (recomendado rodar como `.py`)

Para tempo real estável, rode no terminal:

```bash
python attention_monitor.py
```

**Teclas**: `c` calibra (~1,5s olhando para a tela) e `q` sai.

### Desafio
- Ajuste os limiares (`ear_fechado`, `mar_bocejo`, `yaw_fora_graus`, etc.) e observe o efeito.
- Faça uma versão que salva um log simples (CSV) com timestamp, EAR, MAR e nível.
- Faça uma versão que desenha uma barra de progresso do score (0–100) no canto.
